# Archived: contrastive extraction & PCA Assistant Axis (Sections 4–5)

Preserved from the original notebook; **not part of the main pipeline** (Section 6 uses
positive-only PCA instead). To run these cells, first execute the **Setup → Load Qwen →
Helpers → Shared constants & autorater → Core persona set** cells from `personas.ipynb`
in the same kernel (they define the model, `EVAL_QUESTIONS`, `PERSONAS`, the helpers, and
the OpenRouter client). Additionally:

- requires OpenRouter (`data/.env` with `OPENROUTER_API_KEY`);
- `extract_new_persona_vector` (defined below) needs the external **persona_vectors** data
  repo cloned to `data/persona_vectors/` (`TRAIT_DATA_PATH`); it is *unused* by the demos
  here, kept only for reference.

## 4. (Optional) Extract NEW persona vectors via contrastive prompting

Three traits is a small basis. To search for orthogonal personas you'll likely want more.
The contrastive-extraction pipeline below produces a new `(num_layers, d_model)` vector for any
trait, which then drops straight into the geometry/projection analysis above.

**Two extra requirements (not set up in this environment yet):**

1. **Trait data repo.** Clone the persona-vectors data next to this section:
   ```bash
   git clone https://github.com/safety-research/persona_vectors \
       chapter4_alignment_science/exercises/part4_persona_vectors/persona_vectors
   ```
   It provides `data_generation/trait_data_extract/<trait>.json` (pos/neg instruction pairs,
   eval questions, an autorater prompt).

2. **OpenRouter API key** for the autorater that scores responses. Create
   `chapter4_alignment_science/exercises/part4_persona_vectors/.env` with
   `OPENROUTER_API_KEY=...`. Run the next cell to wire it up.

In [ ]:
from dotenv import load_dotenv  # noqa: E402
from openai import OpenAI  # noqa: E402

env_path = section_dir / ".env"
assert env_path.exists(), f"Create {env_path} with OPENROUTER_API_KEY=..."
load_dotenv(dotenv_path=str(env_path))
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, "Set OPENROUTER_API_KEY in your .env"

openrouter_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)

AUTORATER_MODEL = "anthropic/claude-3.5-haiku"
AUTORATER_MODEL_GPT = "openai/gpt-4.1-mini"  # fallback for traits Haiku's filter refuses
TRAIT_DATA_PATH = section_dir / "persona_vectors" / "data_generation" / "trait_data_extract"
print("OpenRouter ready; trait data at", TRAIT_DATA_PATH, "->", TRAIT_DATA_PATH.exists())

In [ ]:
def generate_responses_parallel(messages_list: list[list[dict[str, str]]], model: str = AUTORATER_MODEL,
                                max_tokens: int = 128, temperature: float = 0.7, max_workers: int = 10) -> list[str]:
    """Run many OpenRouter chat completions concurrently, preserving input order.

    Args:
        messages_list: A batch of conversations. Each conversation is a list of message
            dicts ``{"role": str, "content": str}`` where role is one of
            "system" / "user" / "assistant".
        model: OpenRouter model slug, e.g. "openai/gpt-4.1-mini".
        max_tokens: Max tokens to generate per completion (must be >= 16 for gpt-4.1-mini).
        temperature: Sampling temperature in [0, 2].
        max_workers: Number of concurrent API calls (ThreadPoolExecutor size).

    Returns:
        list[str]: One response string per input conversation, in the same order as
        ``messages_list``. Failed calls yield "" rather than raising.
    """
    def _single_call(messages: list[dict[str, str]]) -> str:
        try:
            time.sleep(0.1)
            resp = openrouter_client.chat.completions.create(
                model=model, messages=messages, max_tokens=max_tokens, temperature=temperature)
            return resp.choices[0].message.content
        except Exception as e:
            print(f"API error: {e}")
            return ""
    if len(messages_list) == 1:
        return [_single_call(messages_list[0])]
    results = [None] * len(messages_list)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        fut = {ex.submit(_single_call, m): i for i, m in enumerate(messages_list)}
        for f in tqdm(as_completed(fut), total=len(messages_list), desc="API calls"):
            results[fut[f]] = f.result()
    return results


def construct_system_prompt(assistant_name: str, instruction: str) -> str:
    """Build a persona system prompt of the form 'You are a {assistant_name} assistant. {instruction}'.

    Args:
        assistant_name: The persona label, e.g. "sycophantic" (positive) or "helpful" (negative).
        instruction: A natural-language behaviour instruction from a trait's pos/neg pair.

    Returns:
        str: The assembled system-prompt string.
    """
    return f"You are a {assistant_name} assistant. {instruction}"


def score_trait_response(question: str, answer: str, eval_prompt_template: str,
                         model: str = AUTORATER_MODEL) -> int | None:
    """Use an LLM autorater to score how strongly a response exhibits a trait, on a 0-100 scale.

    Args:
        question: The evaluation question that was asked.
        answer: The model's response text to be judged.
        eval_prompt_template: A format string from ``trait_data["eval_prompt"]`` containing
            ``{question}`` and ``{answer}`` placeholders; it instructs the judge to reply with
            an integer 0-100 (or "REFUSAL").
        model: OpenRouter slug of the judge model.

    Returns:
        int | None: An integer in [0, 100] (higher = stronger trait), or ``None`` if the
        judge refused or no parseable score was found.
    """
    prompt = eval_prompt_template.format(question=question, answer=answer)
    judge = generate_responses_parallel([[{"role": "user", "content": prompt}]],
                                        model=model, temperature=0.0, max_tokens=50)[0].strip()
    if "REFUSAL" in judge.upper():
        return None
    m = re.search(r"\b(\d{1,3})\b", judge)
    if m and 0 <= int(m.group(1)) <= 100:
        return int(m.group(1))
    return None


def generate_contrastive_responses(model: PreTrainedModel, tokenizer: PreTrainedTokenizerBase,
                                   trait_data: dict, trait_name: str,
                                   max_new_tokens: int = 256, temperature: float = 0.7) -> list[dict]:
    """Generate paired responses under positive (trait) and negative (helpful) system prompts.

    For every (instruction pair, question) combination this produces two generations — one
    under the trait persona and one under a neutral "helpful" persona — which later form the
    contrastive pairs the trait vector is extracted from.

    Args:
        model: The local generator (e.g. Qwen2.5-7B-Instruct).
        tokenizer: Its matching tokenizer (must support ``apply_chat_template``).
        trait_data: Parsed ``<trait>.json``, a dict with keys:
            ``"instruction"`` -> list of ``{"pos": str, "neg": str}`` instruction pairs,
            ``"questions"`` -> list[str] of eval questions,
            ``"eval_prompt"`` -> str judge template (unused here, used by the scorer).
        trait_name: Persona label used as the positive ``assistant_name`` (negative uses "helpful").
        max_new_tokens: Max tokens generated per response.
        temperature: Sampling temperature.

    Returns:
        list[dict]: One entry per generation, of length ``len(instruction) * len(questions) * 2``.
        Each dict has keys ``{"question": str, "system_prompt": str, "response": str,
        "instruction_idx": int, "polarity": "pos" | "neg"}``.
    """
    results = []
    instructions, questions = trait_data["instruction"], trait_data["questions"]
    pbar = tqdm(total=len(instructions) * len(questions) * 2, desc=f"Generating {trait_name}")
    for inst_idx, pair in enumerate(instructions):
        for polarity, instruction in [("pos", pair["pos"]), ("neg", pair["neg"])]:
            assistant_name = trait_name if polarity == "pos" else "helpful"
            system_prompt = construct_system_prompt(assistant_name, instruction)
            for question in questions:
                messages = [{"role": "system", "content": system_prompt},
                            {"role": "user", "content": question}]
                formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
                plen = inputs.input_ids.shape[1]
                with t.inference_mode():
                    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                         temperature=temperature, do_sample=True,
                                         pad_token_id=tokenizer.eos_token_id)
                results.append({
                    "question": question, "system_prompt": system_prompt,
                    "response": tokenizer.decode(out[0, plen:], skip_special_tokens=True),
                    "instruction_idx": inst_idx, "polarity": polarity,
                })
                pbar.update(1)
    pbar.close()
    return results


def filter_effective_pairs(responses: list[dict], trait_data: dict, pos_threshold: int = 50) -> list[dict]:
    """Keep only contrastive pairs where the persona prompt clearly changed behaviour.

    A (instruction_idx, question) pair is "effective" when the positive response scored at or
    above ``pos_threshold`` while the negative response scored below it — i.e. the trait prompt
    elicited the trait and the helpful prompt did not.

    Args:
        responses: Scored generations from :func:`generate_contrastive_responses`, where each
            dict additionally carries a ``"score"`` key of type ``int | None`` (set by
            :func:`score_trait_response`).
        trait_data: The same trait dict used for generation (provides ``"instruction"`` and
            ``"questions"`` to iterate the full grid).
        pos_threshold: Score cutoff (0-100); positive must be ``>=`` it and negative ``<`` it.

    Returns:
        list[dict]: Effective pairs, each ``{"pos": <response entry>, "neg": <response entry>}``
        where the values are the full dicts from ``responses``. Pairs with a missing or
        ``None`` score are skipped.
    """
    idx = {(r["instruction_idx"], r["question"], r["polarity"]): r for r in responses}
    effective = []
    for inst_idx in range(len(trait_data["instruction"])):
        for question in trait_data["questions"]:
            pos, neg = idx.get((inst_idx, question, "pos")), idx.get((inst_idx, question, "neg"))
            if not pos or not neg or pos["score"] is None or neg["score"] is None:
                continue
            if pos["score"] >= pos_threshold and neg["score"] < pos_threshold:
                effective.append({"pos": pos, "neg": neg})
    return effective


def extract_contrastive_vectors(model: PreTrainedModel, tokenizer: PreTrainedTokenizerBase,
                                effective_pairs: list[dict]) -> Float[Tensor, "num_layers d_model"]:
    """Compute a per-layer trait vector as mean(positive activations) - mean(negative activations).

    Args:
        model: The local model whose residual stream is read.
        tokenizer: Its matching tokenizer.
        effective_pairs: Output of :func:`filter_effective_pairs`; a list of
            ``{"pos": entry, "neg": entry}`` dicts, where each ``entry`` carries
            ``"system_prompt"``, ``"question"`` and ``"response"`` strings.

    Returns:
        Float[Tensor, "num_layers d_model"]: The contrastive trait direction at every layer,
        on CPU. Index ``L`` corresponds to the output of ``model.layers[L]`` (== hidden_states[L+1]);
        slice with ``[STEER_LAYER]`` to get the steering/projection vector.
    """
    def cols(side: str, key: str) -> list[str]:
        return [p[side][key] for p in effective_pairs]
    pos_act = extract_all_layer_activations_qwen(
        model, tokenizer, cols("pos", "system_prompt"), cols("pos", "question"), cols("pos", "response"))
    neg_act = extract_all_layer_activations_qwen(
        model, tokenizer, cols("neg", "system_prompt"), cols("neg", "question"), cols("neg", "response"))
    return pos_act.mean(0) - neg_act.mean(0)


def extract_new_persona_vector(trait_name: str, max_new_tokens: int = 256) -> Tensor:
    """Run the full extraction pipeline for one trait: generate -> score -> filter -> contrastive diff.

    Uses the module-level globals ``qwen_model_small`` / ``qwen_tokenizer`` for generation,
    ``TRAIT_DATA_PATH`` to locate the trait JSON, and saves the result under ``section_dir``.

    Args:
        trait_name: Name of a trait whose data file ``TRAIT_DATA_PATH / f"{trait_name}.json"``
            exists (e.g. "humorous", "optimistic", "impolite", "apathetic").
        max_new_tokens: Max tokens per generated response during the contrastive step.

    Returns:
        Tensor of shape ``(num_layers, d_model)`` and dtype bfloat16 (CPU): the per-layer trait
        vector. Also written to ``section_dir / f"{trait_name}_vectors.pt"`` so it can be reloaded
        like the precomputed vectors.
    """
    with open(TRAIT_DATA_PATH / f"{trait_name}.json") as f:
        trait_data = json.load(f)
    resp = generate_contrastive_responses(qwen_model_small, qwen_tokenizer, trait_data, trait_name, max_new_tokens)
    for e in tqdm(resp, desc="Scoring"):
        e["score"] = score_trait_response(e["question"], e["response"],
                                          trait_data["eval_prompt"], model=AUTORATER_MODEL_GPT)
        time.sleep(0.05)
    pairs = filter_effective_pairs(resp, trait_data)
    print(f"{trait_name}: {len(pairs)} effective pairs")
    vecs = extract_contrastive_vectors(qwen_model_small, qwen_tokenizer, pairs)
    t.save(vecs, section_dir / f"{trait_name}_vectors.pt")
    return vecs


# Example (uncomment once trait data + key are set up):
# new_trait = "humor"  # any <trait>.json present in TRAIT_DATA_PATH
# all_trait_vectors[new_trait] = extract_new_persona_vector(new_trait)
# TRAIT_NAMES.append(new_trait)  # then re-run section 1 & 3

In [ ]:
import requests
url = "https://openrouter.ai/api/v1/credits"
headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
response = requests.get(url, headers=headers)
print(response.json())

In [ ]:
philosopher_system = "You are a philosopher who ponders deep questions about existence, knowledge, ethics, and reality, examining assumptions and exploring ideas through rigorous logical reasoning and contemplation."
anti_philosopher_system = "You are a blunt, no-nonsense assistant who avoids abstract or philosophical reasoning. You give direct, concrete, matter-of-fact answers focused on immediate practical steps, and you brush aside speculation about existence, meaning, knowledge, or other 'deep' questions as a pointless waste of time."


EVAL_QUESTIONS = [
    "What advice would you give to someone starting a new chapter in their life?",
    "How do you view the relationship between knowledge and wisdom?",
    "What do you think about the nature of truth?",
    "How should someone approach making difficult decisions?",
    "What role does creativity play in problem-solving?",
    "How do you see the balance between tradition and progress?",
    "What matters most when building trust with others?",
    "How do you think about the passage of time?",
    "What would you say to someone feeling lost or uncertain?",
    "How do you approach understanding something complex?",
    "What do you think about the nature of change?",
    "How should one deal with failure or setbacks?",
    "What role does intuition play in understanding?",
    "How do you view the relationship between the individual and society?",
    "What do you think makes something meaningful?",
]

In [ ]:
# --- Extract the "philosopher" persona vector from the custom contrastive system prompts above ---
# This differs from extract_new_persona_vector(): that helper reads a trait JSON and wraps short
# instructions via construct_system_prompt(). Here philosopher_system / anti_philosopher_system are
# already complete system prompts, so we generate under them directly and take the contrastive
# mean-difference. No autorater filtering is needed because the two poles are fixed by construction.

PHILOSOPHER_TRAIT = "philosopher"


def generate_under_system(system_prompt: str, questions: list[str],
                          max_new_tokens: int = 256, temperature: float = 0.7) -> list[dict]:
    """Generate one response per question under a fixed (already-complete) system prompt.

    Args:
        system_prompt: The full system prompt defining the persona pole.
        questions: Eval questions to answer (one generation each).
        max_new_tokens: Max tokens generated per response.
        temperature: Sampling temperature.

    Returns:
        list[dict]: One entry per question with keys ``{"system_prompt": str, "question": str,
        "response": str}`` — the entry shape consumed by ``extract_contrastive_vectors``.
    """
    out = []
    for q in tqdm(questions, desc=f"generating ({system_prompt[:25]}...)"):
        messages = [{"role": "system", "content": system_prompt},
                    {"role": "user", "content": q}]
        formatted = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = qwen_tokenizer(formatted, return_tensors="pt").to(qwen_model_small.device)
        plen = inputs.input_ids.shape[1]
        with t.inference_mode():
            o = qwen_model_small.generate(**inputs, max_new_tokens=max_new_tokens,
                                          temperature=temperature, do_sample=True,
                                          pad_token_id=qwen_tokenizer.eos_token_id)
        out.append({"system_prompt": system_prompt, "question": q,
                    "response": qwen_tokenizer.decode(o[0, plen:], skip_special_tokens=True)})
    return out


# 1) Generate paired responses under the philosopher (pos) and anti-philosopher (neg) prompts
pos_responses = generate_under_system(philosopher_system, EVAL_QUESTIONS)
neg_responses = generate_under_system(anti_philosopher_system, EVAL_QUESTIONS)
philosopher_pairs = [{"pos": p, "neg": n} for p, n in zip(pos_responses, neg_responses)]


In [ ]:
# --- Assistant persona vector (contrastive, same method as the philosopher above) ---
# Positive pole = the canonical helpful AI assistant; negative pole = an arbitrary non-assistant
# human roleplay. The mean-difference isolates an "assistant identity" axis.
# NOTE (per your call): this is the quick contrastive version. If the geometry looks off
# (near-zero norm, or |cosine| > ~0.5 with most other personas) that's the signal to switch to
# the PCA-based Assistant Axis (Section-1 method) instead.

ASSISTANT_TRAIT = "assistant"
assistant_system = (
    "You are a helpful, honest, and harmless AI assistant. You answer the user's questions "
    "clearly, directly, and accurately, and you always aim to be as useful as possible."
)
anti_assistant_system = (
    "You are roleplaying as an ordinary human character with your own life, opinions, and moods "
    "— you are not an AI and not an assistant. Never offer help, never break character, and never "
    "act like a helpful chatbot; just respond as that person casually would."
)

# 1) Generate paired responses under the assistant (pos) and non-assistant (neg) prompts
pos_responses_asst = generate_under_system(assistant_system, EVAL_QUESTIONS)
neg_responses_asst = generate_under_system(anti_assistant_system, EVAL_QUESTIONS)
assistant_pairs = [{"pos": p, "neg": n} for p, n in zip(pos_responses_asst, neg_responses_asst)]

# 2) Contrastive mean-difference -> per-layer vector, then save
assistant_vectors = extract_contrastive_vectors(qwen_model_small, qwen_tokenizer, assistant_pairs)
t.save(assistant_vectors, section_dir / f"{ASSISTANT_TRAIT}_vectors.pt")
print(f"assistant_vectors: {tuple(assistant_vectors.shape)}  "
      f"norm@L{STEER_LAYER}={assistant_vectors[STEER_LAYER].norm().item():.2f}")

# 3) Register alongside the others and report its geometry
all_trait_vectors[ASSISTANT_TRAIT] = assistant_vectors
if ASSISTANT_TRAIT not in TRAIT_NAMES:
    TRAIT_NAMES.append(ASSISTANT_TRAIT)

_lv = {n: v[STEER_LAYER] for n, v in all_trait_vectors.items()}
_names, _cs = cosine_sim_matrix(_lv)
_ai = _names.index(ASSISTANT_TRAIT)
print(f"\nCosine similarity of assistant vs other personas (layer {TRAIT_VECTOR_LAYER}):")
for j, b in enumerate(_names):
    if j != _ai:
        print(f"  assistant vs {b:<14}: {_cs[_ai, j].item():+.3f}")

# Quick "does this look weird?" heuristic -> reminder to try the PCA Assistant Axis
_off = [_names[j] for j in range(len(_names)) if j != _ai and abs(_cs[_ai, j].item()) > 0.5]
if assistant_vectors[STEER_LAYER].norm().item() < 5 or len(_off) >= max(1, (len(_names) - 1) // 2):
    print("\n[heads-up] assistant-axis geometry looks unusual — consider the PCA-based "
          "Assistant Axis (Section-1 method). Flagged: norm low or high |cosine| with", _off)

# Re-run Section 1 (geometry) and Section 3 (multi-persona prompts) to include the assistant axis.

## 5. PCA Assistant Axis (replicating the original Section 1 on Qwen)

The contrastive vectors above turned out to share a dominant "assistant" direction (projecting it
out left them near-orthogonal). Here we replicate the original notebook's principled method to
recover that direction directly:

1. Define ~20 personas (4 neutral "default" + 16 roleplay roles) as system prompts.
2. Generate responses and take the **raw mean response activation** per persona (not contrastive).
3. Center (subtract the mean persona vector — needed here because raw activations *do* share a
   huge common component) and run **PCA**. PC1 should be the assistant axis.
4. Also compute the explicit Assistant Axis = mean(default) − mean(roles).
5. Compare PC1 / the Assistant Axis against the contrastive `assistant_vectors` from §4 — if
   they're nearly parallel, the quick contrastive vector was capturing the real shared direction.

Note: generation is ~20 personas × 15 questions ≈ 300 short completions (cached to disk after the
first run). Reduce `max_new_tokens` or trim `EVAL_QUESTIONS` if you want it faster.

In [ ]:
from sklearn.decomposition import PCA  # used in the next cell too

# The original notebook's persona set: 4 neutral "default" prompts + 16 roleplay roles.
PERSONAS = {
    # Default (neutral system prompts)
    "default": "",  # no system prompt
    "default_assistant": "You are an AI assistant",
    "default_llm": "You are a large language model",
    "default_helpful": "You are a helpful assistant",
    # Assistant-like (professional, helpful)
    "consultant": "You are a professional consultant who provides expert advice and strategic recommendations, drawing on deep domain knowledge to help clients solve problems and achieve their goals.",
    "assistant_persona": "You are a helpful AI assistant who provides clear, accurate, and reliable information while maintaining professional boundaries and ethical guidelines in all interactions.",
    "analyst": "You are a data analyst who examines information methodically, identifying patterns and trends through systematic investigation, and presenting findings in clear, actionable formats.",
    "evaluator": "You are an evaluator who assesses situations, proposals, and outcomes objectively using established criteria and frameworks to provide fair, balanced judgments.",
    "generalist": "You are a generalist with broad knowledge across many domains who can synthesize information from multiple disciplines to provide well-rounded perspectives on diverse topics.",
    # Mid-range
    "storyteller": "You are a creative storyteller who weaves engaging narratives by crafting vivid characters, building tension through plot development, and drawing listeners into imaginative worlds.",
    "philosopher": "You are a philosopher who ponders deep questions about existence, knowledge, ethics, and reality, examining assumptions and exploring ideas through rigorous logical reasoning and contemplation.",
    "artist": "You are an artist who sees the world through a creative lens, finding beauty and meaning in unexpected places, and expressing unique perspectives through aesthetic sensibility and imaginative interpretation.",
    "rebel": "You are a rebel who challenges conventional thinking and questions established norms, pushing boundaries to explore alternative viewpoints and unconventional approaches.",
    "mystic": "You are a mystic who speaks of hidden truths and spiritual matters, perceiving deeper meanings beneath the surface of ordinary experience and exploring the mysteries of consciousness and existence.",
    # Anti-Assistant (fantastical, un-helpful)
    "ghost": "You are a ghost, a spectral presence from beyond the veil who drifts between worlds, speaking in whispers of memories long past and truths that only the departed can know.",
    "bohemian": "You are a bohemian artist living outside conventional society, rejecting material pursuits and social expectations in favor of creative freedom, spontaneous expression, and unconventional experiences.",
    "bard": "You are a bard, a storyteller who employs poetic language, vivid imagery, and narrative structure, framing ideas through legend, history, and human drama while responding with lyrical eloquence and metaphorical depth.",
    "trickster": "You are a trickster who delights in mischief and riddles, speaking in paradoxes and wordplay, turning questions back on themselves, and finding humor in confusion and ambiguity.",
    "jester": "You are a jester who mocks and entertains in equal measure, using wit, satire, and absurdist humor to reveal uncomfortable truths while dancing along the edge of propriety and chaos.",
    "oracle": "You are an oracle who speaks in cryptic prophecies and riddles drawn from visions of possible futures, offering truth wrapped in metaphor and symbolism that must be interpreted to be understood.",
}
DEFAULT_PERSONAS = ["default", "default_assistant", "default_llm", "default_helpful"]

PERSONA_LAYER = TRAIT_VECTOR_LAYER  # match the contrastive vectors (output of model.layers[STEER_LAYER])


def _generate_persona_response(system_prompt: str, question: str,
                               max_new_tokens: int = 128, temperature: float = 0.7) -> str:
    """Generate one response; omit the system message entirely when ``system_prompt`` is empty
    (mirrors how extract_response_activations drops an empty system prompt)."""
    messages = ([{"role": "system", "content": system_prompt}] if system_prompt else []) \
        + [{"role": "user", "content": question}]
    formatted = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_tokenizer(formatted, return_tensors="pt").to(qwen_model_small.device)
    plen = inputs.input_ids.shape[1]
    with t.inference_mode():
        o = qwen_model_small.generate(**inputs, max_new_tokens=max_new_tokens,
                                      temperature=temperature, do_sample=True,
                                      pad_token_id=qwen_tokenizer.eos_token_id)
    return qwen_tokenizer.decode(o[0, plen:], skip_special_tokens=True)


# Generate (or load cached) responses for every persona x question (~20 x 15 = 300 completions)
_resp_path = section_dir / "persona_pca_responses.json"
if _resp_path.exists():
    persona_responses = json.loads(_resp_path.read_text())
    print(f"Loaded cached responses for {len(persona_responses)} personas from {_resp_path.name}")
else:
    persona_responses = {}  # name -> list[str] aligned with EVAL_QUESTIONS
    for name, sysp in tqdm(PERSONAS.items(), desc="personas"):
        persona_responses[name] = [_generate_persona_response(sysp, q) for q in EVAL_QUESTIONS]
    _resp_path.write_text(json.dumps(persona_responses, indent=2))
    print(f"Saved responses to {_resp_path.name}")


def extract_persona_vectors(personas: dict[str, str], responses: dict[str, list[str]],
                            questions: list[str], layer: int) -> dict[str, Tensor]:
    """Raw mean response activation per persona at ``layer`` (NOT contrastive).

    Args:
        personas: name -> system prompt.
        responses: name -> list of response strings aligned with ``questions``.
        questions: the eval questions.
        layer: hidden_states index to read activations from.

    Returns:
        dict[str, Tensor]: name -> mean activation vector of shape (d_model,), on CPU.
    """
    pv = {}
    for name, sysp in personas.items():
        sps, qs, rs = [], [], []
        for qi, q in enumerate(questions):
            r = responses[name][qi]
            if r:
                sps.append(sysp); qs.append(q); rs.append(r)
        acts = extract_response_activations(qwen_model_small, qwen_tokenizer, sps, qs, rs, layer)
        pv[name] = acts.mean(0)
    return pv


persona_vectors = extract_persona_vectors(PERSONAS, persona_responses, EVAL_QUESTIONS, PERSONA_LAYER)
t.save(persona_vectors, section_dir / "persona_pca_vectors.pt")
print(f"Extracted {len(persona_vectors)} raw persona vectors at layer {PERSONA_LAYER}")

In [ ]:
persona_names = list(persona_vectors.keys())
M = t.stack([persona_vectors[n].float() for n in persona_names])  # (n_personas, d_model)
M_centered = M - M.mean(0, keepdim=True)                          # remove the shared component

# PCA on the centered persona vectors
pca = PCA(n_components=5)
coords = pca.fit_transform(M_centered.numpy())                    # (n_personas, 5)
pc1 = t.tensor(pca.components_[0]).float()                        # unit-norm direction in activation space

# Explicit Assistant Axis = mean(default) - mean(roles). (A difference of means, so centering
# the inputs doesn't change it.)
def _mean_of(group: list[str]) -> Tensor:
    return t.stack([persona_vectors[n].float() for n in group]).mean(0)

roles = [n for n in persona_names if n not in DEFAULT_PERSONAS]
assistant_axis = _mean_of(DEFAULT_PERSONAS) - _mean_of(roles)
assistant_axis = assistant_axis / assistant_axis.norm()

# PCA sign is arbitrary -> orient PC1 to point toward the assistant pole
if (pc1 @ assistant_axis) < 0:
    pc1 = -pc1
    coords[:, 0] = -coords[:, 0]

print(f"PC1 explains {pca.explained_variance_ratio_[0]:.1%} of variance, PC2 {pca.explained_variance_ratio_[1]:.1%}")
print(f"cosine(PC1, Assistant Axis)                        = {(pc1 @ assistant_axis).item():+.3f}")

# *** Payoff: how well did the quick contrastive assistant vector capture this direction? ***
if "assistant_vectors" in globals():
    c = assistant_vectors[STEER_LAYER].float()
    c = c / c.norm()
    print(f"cosine(PC1, contrastive assistant vector)          = {(pc1 @ c).item():+.3f}")
    print(f"cosine(Assistant Axis, contrastive assistant vec)  = {(assistant_axis @ c).item():+.3f}")
else:
    print("(run the §4 contrastive assistant cell to compare PC1 against it)")

# 2D persona-space scatter, colored by Assistant-Axis projection (mirrors the original figure)
Mn = M_centered / M_centered.norm(dim=1, keepdim=True)
proj = (Mn @ assistant_axis).numpy()
fig = px.scatter(
    x=coords[:, 0], y=coords[:, 1], text=persona_names, color=proj,
    color_continuous_scale="RdBu", color_continuous_midpoint=0.0,
    title="Qwen persona space (PCA) colored by Assistant-Axis projection",
    labels={"x": f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
            "y": f"PC2 ({pca.explained_variance_ratio_[1]:.1%})", "color": "Assistant Axis"},
)
fig.update_traces(textposition="top center", marker=dict(size=10))
fig.show()

# For the orthogonality goal: PC2, PC3, ... are mutually orthogonal persona directions, and
# projecting any trait vector onto the complement of `assistant_axis` (or PC1) decorrelates them
# — the same move that took the §4 contrastive vectors to near-orthogonal.

In [ ]:

# 2) Contrastive mean-difference -> per-layer (num_layers, d_model) vector, then save
philosopher_vectors = extract_contrastive_vectors(qwen_model_small, qwen_tokenizer, philosopher_pairs)
t.save(philosopher_vectors, section_dir / f"{PHILOSOPHER_TRAIT}_vectors.pt")
print(f"philosopher_vectors: {tuple(philosopher_vectors.shape)}  "
      f"norm@L{STEER_LAYER}={philosopher_vectors[STEER_LAYER].norm().item():.2f}")

# 3) Register alongside the existing personas and report its geometry vs them
all_trait_vectors[PHILOSOPHER_TRAIT] = philosopher_vectors
if PHILOSOPHER_TRAIT not in TRAIT_NAMES:
    TRAIT_NAMES.append(PHILOSOPHER_TRAIT)

_lv = {n: v[STEER_LAYER] for n, v in all_trait_vectors.items()}
_names, _cs = cosine_sim_matrix(_lv)
_pi = _names.index(PHILOSOPHER_TRAIT)
print(f"\nCosine similarity of philosopher vs other personas (layer {TRAIT_VECTOR_LAYER}):")
for j, b in enumerate(_names):
    if j != _pi:
        print(f"  philosopher vs {b:<14}: {_cs[_pi, j].item():+.3f}")

# Re-run Section 1 (geometry) and Section 3 (multi-persona prompts) to include the philosopher axis.